# Day 2 · Final pipeline on our own photos

Run cells **top to bottom**. 💰 = calls the AI and spends credits (skips work that already exists).

| # | What | Cost |
|---|---|---|
| 1 | Settings (always first) | free |
| 2 | fal.ai key | free |
| 3 | List of our photos (front + 2 details) | free |
| 4 | Test on one garment | 💰 $0.18 |
| 5 | Look at that garment | free |
| 6 | All 12 garments + fill g01/g11/g15 | 💰 $2.20 |
| 7 | Look at all 15 | free |
| 8 | Ours vs theirs comparison (report picture) | free |
| 9 | Extra garments x01–x03 (demo only) | 💰 $0.55 |
| 10 | Assemble hand-in (27 dev + 15 test) + validate | free |
| 11 | Fidelity check vs our own photos | free |
| 12 | Dev leaderboard v2 (27 dev through final pipeline) | 💰 $4.90 |
| 13 | Refinement: manifest with ALL close-ups | free |
| 14 | Regenerate our 12 garments with notes | 💰 $2.20 |
| 15 | Same for g01 and g15 | 💰 $0.40 |
| 16 | Look at every candidate, force picks | free |
| 17 | Rebuild hand-in (cell 10) | free |

## 1. Settings (always first)

In [ ]:
import os, sys, json, csv, subprocess, getpass
from pathlib import Path
from PIL import Image
from IPython.display import display

REPO = Path.home() / "Desktop" / "Neural Wave" / "neural-wave-2026"
if not Path("scorer").is_dir():
    os.chdir(REPO)
PY   = str(Path.cwd() / ".venv/bin/python")
DATA = Path("../sartiq_neural_wave_2026")
CAP  = Path("../captures")
TEAM = "aactivetion_function"
assert Path("pipeline.py").exists(), "pipeline.py is missing from the repo folder"
assert (CAP / "day1_clean/best_fronts.csv").exists(), "captures/day1_clean/best_fronts.csv not found"

def run(*args):
    cmd = [str(a) for a in args]
    print("$", " ".join(cmd), "\n")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "UserWarning" in line or "warnings.warn" in line or "HF_TOKEN" in line:
            continue
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        print(f"\n⚠️ finished with error code {p.returncode}")
    return p.returncode

def strip(paths, h=320):
    ims = []
    for p in paths:
        if p and Path(p).exists():
            im = Image.open(p).convert("RGB")
            ims.append(im.resize((max(1, int(im.width * h / im.height)), h)))
    canvas = Image.new("RGB", (sum(i.width for i in ims) + 10 * (len(ims) - 1), h), "white")
    x = 0
    for i in ims:
        canvas.paste(i, (x, 0)); x += i.width + 10
    return canvas

def find_pred(root, team, gid):
    for ext in ("png", "jpg", "jpeg", "webp"):
        p = Path(root) / team / gid / f"front.{ext}"
        if p.exists():
            return p

print("Working folder:", Path.cwd()); print("All good ✅")

## 2. The fal.ai key (paste it in the box that appears, never in the code)

In [ ]:
if not os.environ.get("FAL_KEY"):
    os.environ["FAL_KEY"] = getpass.getpass("Paste the fal.ai key: ").strip()
print("Key set ✅")

## 3. Build the list of our photos (free)

In [ ]:
run(PY, "make_ours_manifest.py", "--captures", CAP, "--out", CAP / "ours.jsonl", "--details", "2")
OURS = [json.loads(l) for l in open(CAP / "ours.jsonl")]
print([r["garment_id"] for r in OURS])

## 4. 💰 Test on ONE garment (≈ $0.18, ≈ 1 min)

In [ ]:
run(PY, "pipeline.py", "--manifest", CAP / "ours.jsonl", "--dataset", CAP,
    "--out", "submissions_final", "--team", TEAM, "--candidates", "3", "--max-inputs", "3", "--limit", "1")

## 5. Look at it: our photo | the 3 AI candidates | the one the pipeline kept

In [ ]:
def show_garment(gid):
    row = next(r for r in OURS if r["garment_id"] == gid)
    cands = sorted(Path("submissions_final/_candidates", TEAM, gid).glob("cand_*.png"))
    print(gid, "→ our photo | candidates | kept")
    display(strip([CAP / row["inputs"][0]] + cands + [find_pred("submissions_final", TEAM, gid)]))

show_garment(OURS[0]["garment_id"])

## 6. 💰 All 12 garments + fill g01/g11/g15 from the organisers' photos (≈ $2.20, ≈ 15 min)

In [ ]:
run(PY, "pipeline.py", "--manifest", CAP / "ours.jsonl", "--dataset", CAP,
    "--out", "submissions_final", "--team", TEAM, "--candidates", "3", "--max-inputs", "3",
    "--fill-from", Path("submissions") / TEAM, "--fill-ids", DATA / "test/inputs.jsonl")

## 7. Look at all 15 (free). Note any bad one for the Quality checker.

In [ ]:
for gid in sorted(p.name for p in Path("submissions_final", TEAM).iterdir()):
    if any((Path("submissions_final/_candidates", TEAM, gid)).glob("cand_*.png")):
        show_garment(gid)
    else:
        print(gid, "→ organisers' photo | kept (fallback)")
        row = next(json.loads(l) for l in open(DATA / "test/inputs.jsonl") if json.loads(l)["garment_id"] == gid)
        display(strip([DATA / row["inputs"][0], find_pred("submissions_final", TEAM, gid)]))

## 8. Compare: our photo vs the organisers' photo, same garment, same AI (free)
This is the "capture is the bottleneck" picture for the report and the video.

In [ ]:
def compare(gid):
    row_o = next(json.loads(l) for l in open(DATA / "test/inputs.jsonl") if json.loads(l)["garment_id"] == gid)
    row_u = next(r for r in OURS if r["garment_id"] == gid)
    print(gid, "→ their photo | AI from their photo || our photo | AI from our photo")
    display(strip([DATA / row_o["inputs"][0], find_pred("submissions", TEAM, gid),
                   CAP / row_u["inputs"][0], find_pred("submissions_final", TEAM, gid)], h=300))

for gid in ("g02", "g05", "g09", "g12"):
    compare(gid)

## 9. 💰 The three extra garments x01–x03, for the demo only (≈ $0.55). They never go into submissions.

In [ ]:
run(PY, "make_ours_manifest.py", "--captures", CAP, "--out", CAP / "extras.jsonl", "--details", "2",
    "--alias", "x01=g91", "x02=g92", "x03=g93")
run(PY, "pipeline.py", "--manifest", CAP / "extras.jsonl", "--dataset", CAP,
    "--out", "demo_extras", "--team", TEAM, "--candidates", "3", "--max-inputs", "3")
for gid in ("g91", "g92", "g93"):
    display(strip([find_pred("demo_extras", TEAM, gid)], h=260))

## 10. Assemble the hand-in: 27 dev + 15 test in one folder, validate (free)
Dev pictures come from the earlier run (organisers' photos). Test pictures: **ours** by default, or **theirs** per garment as listed in `final_choice.csv` (one line per exception, e.g. `g05,theirs`).

In [ ]:
import shutil
CHOICE = Path("final_choice.csv")
if not CHOICE.exists():
    CHOICE.write_text("gid,source\n")          # add lines like: g05,theirs
choice = {r["gid"].strip(): r["source"].strip() for r in csv.DictReader(open(CHOICE)) if r.get("gid")}

HANDIN = Path("handin") / TEAM
HANDIN.mkdir(parents=True, exist_ok=True)
report = []
for split in ("dev", "test"):
    for l in open(DATA / split / "inputs.jsonl"):
        gid = json.loads(l)["garment_id"]
        if split == "dev":
            src, source = find_pred("submissions", TEAM, gid), "theirs"
        else:
            source = choice.get(gid, "ours")
            src = find_pred("submissions_final" if source == "ours" else "submissions", TEAM, gid)
        assert src, f"{gid}: no picture for source={source}"
        dst = HANDIN / gid
        dst.mkdir(exist_ok=True)
        for old in dst.glob("front.*"):
            old.unlink()
        shutil.copy(src, dst / src.name)
        if split == "test":
            report.append(f"{gid}:{source}")
print("test choices:", "  ".join(report))
print("copied", len(list(HANDIN.iterdir())), "garments ->", HANDIN)
run(PY, "scripts/make_submission.py", "--submissions", "handin", "--team", TEAM, "--dataset", DATA)

## 11. Fidelity check: how close is each AI picture to OUR clean photo? (free, ≈ 3 min)
Not the leaderboard score (that needs Sartiq's secret references). It ranks garments and shows the scorer's warning flags. Low score or a non-ok status = look at that garment.

In [ ]:
import shutil, pandas as pd
refs = Path("fidelity_refs")
for r in OURS:
    d = refs / r["garment_id"]; d.mkdir(parents=True, exist_ok=True)
    shutil.copy(CAP / r["inputs"][0], d / "front.jpg")
theirs = Path("fidelity_subs_theirs")
if not (theirs / TEAM).exists():
    shutil.copytree(Path("submissions") / TEAM, theirs / TEAM)
for name, subs in (("ours", "submissions_final"), ("theirs", theirs)):
    out = Path(f"fidelity_{name}")
    if (out / "scores.jsonl").exists():
        shutil.rmtree(out)                      # always re-score the current pictures
    run(PY, "-m", "sartiq_score.cli", "batch", "--refs", refs, "--subs", subs, "--out", out, "--jobs", "1")

def load_scores(folder):
    rows = []
    for l in open(Path(folder) / "scores.jsonl"):
        r = json.loads(l)
        if r.get("team") not in (TEAM, TEAM.upper(), "AACtivEtion_Function"):
            continue
        sub = r.get("subscores") or {}
        rows.append({"garment": r.get("garment"), "score": r.get("score"), "status": r.get("status"),
                     "flags": "|".join(r.get("flags") or []), "colour": sub.get("colour"),
                     "lpips": sub.get("lpips"), "shape": sub.get("geometry")})
    return pd.DataFrame(rows)

a, b = load_scores("fidelity_ours"), load_scores("fidelity_theirs")
m = a.merge(b, on="garment", how="left", suffixes=("_ours", "_theirs"))
pd.set_option("display.width", 220)
display(m.sort_values("score_ours").round(3))

## 12. 💰 Dev leaderboard, version 2: the 27 dev garments through the final pipeline (≈ $4.90, ≈ 6 min)
Then upload `dev_submission_v2.zip` on the submit page.

In [ ]:
import zipfile
run(PY, "pipeline.py", "--manifest", DATA / "dev/inputs.jsonl", "--dataset", DATA,
    "--out", "submissions_dev_v2", "--team", TEAM, "--candidates", "3", "--max-inputs", "3")
dev_ids = [json.loads(l)["garment_id"] for l in open(DATA / "dev/inputs.jsonl")]
with zipfile.ZipFile("dev_submission_v2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for gid in dev_ids:
        src = find_pred("submissions_dev_v2", TEAM, gid)
        assert src, f"{gid} missing"
        z.write(src, f"{gid}/{src.name}")
print("dev_submission_v2.zip ready ->", Path("dev_submission_v2.zip").resolve())

## 13. Refinement round: ALL close-ups as input + per-garment instructions (free step first)
`garment_notes.json` holds the extra instructions per garment (plain English, edit freely). New candidates are named `cand_r0..r2`; the old ones stay, the pick is made among all of them.

In [ ]:
run(PY, "make_ours_manifest.py", "--captures", CAP, "--out", CAP / "ours_all.jsonl")
OURS = [json.loads(l) for l in open(CAP / "ours_all.jsonl")]
print({r["garment_id"]: len(r["inputs"]) for r in OURS})
if not Path("manual_picks.csv").exists():
    Path("manual_picks.csv").write_text("gid,file\n")     # add lines like: g07,cand_1.png

## 14. 💰 Regenerate our 12 garments with all close-ups + notes (≈ $2.20, ≈ 6 min)

In [ ]:
run(PY, "pipeline.py", "--manifest", CAP / "ours_all.jsonl", "--dataset", CAP,
    "--out", "submissions_final", "--team", TEAM, "--candidates", "3", "--max-inputs", "8",
    "--tag", "r", "--notes", "garment_notes.json", "--picks", "manual_picks.csv")

## 15. 💰 Same for g01 and g15 (organisers' photos, ≈ $0.40)

In [ ]:
run(PY, "pipeline.py", "--manifest", DATA / "test/inputs.jsonl", "--dataset", DATA,
    "--out", "submissions_final", "--team", TEAM, "--candidates", "3", "--max-inputs", "8",
    "--only", "g01,g15", "--tag", "r", "--notes", "garment_notes.json", "--picks", "manual_picks.csv")

## 16. Look at every candidate of one garment, with file names (free)
To force a candidate, add a line `gXX,cand_r1.png` to `manual_picks.csv` and rerun cell 14 (or 15): nothing is regenerated, it only re-picks.

In [ ]:
def show_all(gid):
    cands = sorted(Path("submissions_final/_candidates", TEAM, gid).glob("cand_*.png"))
    kept = find_pred("submissions_final", TEAM, gid)
    kept_name = next((c.name for c in cands if c.read_bytes() == kept.read_bytes()), "?")
    row = next((r for r in OURS if r["garment_id"] == gid), None)
    if row:
        print(f"===== {gid}: our photos"); display(strip([CAP / p for p in row["inputs"]], h=260))
    print(f"===== {gid}: candidates   (kept = {kept_name})")
    for c in cands:
        print(c.name + ("   <-- KEPT" if c.name == kept_name else "")); display(strip([c], h=420))

show_all("g01")

## 17. When happy: run cell 10 again to rebuild the hand-in folder and validate it.